# 🐛 Bug Fixes Applied

## Fix 1: ROI Display in Cropped Images

**Problem 1:** ROIs were not displayed when cropping was applied because the code had `if roi_collection is not None and not crop_applied:` which completely skipped ROI plotting for cropped images.

**Problem 2:** When cropping + flipping was applied, ROI coordinates were flipped using the **full image dimensions** instead of mirroring within the **crop extent range**.

**Solution:** 
1. Removed the `and not crop_applied` condition so ROIs are always processed
2. Added logic to filter ROIs that fall within crop boundaries
3. Fixed coordinate transformation to mirror ROIs within the crop extent when flipping

---

## Fix 2: Aspect Ratio Distortion in ALL Crops (RGB + Wavelength)

**Problem:** Cropped images were rectangular/distorted instead of square because:
1. `tight_layout()` was being called for crops, which overrode the aspect ratio
2. Aspect ratio fix was only applied to wavelength colormap mode, not RGB crops

**Solution:**
1. **Extent uses relative coordinates** (0 to width, 0 to height) instead of absolute coordinates
2. **Aspect ratio preservation** (`ax.set_aspect('equal')`) now applies to ALL crops (RGB + wavelength)
3. **Skip `tight_layout()`** for all crops to prevent aspect ratio from being undone
4. Track/Slit axis labels now show relative indices (0 to crop size)

---

**📋 Testing Steps:**
1. Run cell 3 below to **reload the module**
2. Run your RGB crop (highlighted code) - should now be square
3. Test with ROIs - should display correctly with square aspect
4. Check axis labels - should show relative coordinates (0 to width/height)

In [1]:
# Import required libraries
import importlib
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add parent directory to path
sys.path.append(os.path.abspath("../"))

# Import gref4hsi modules
from utils.gref_pipeline import georef
from gref_pipeline import config

importlib.reload(georef)
from utils.gref_pipeline.georef import *

# Import NDI analysis utilities
from utils.ndi_analysis_utils import *

In [2]:
# 🔄 Reload module - WITH GLOBAL COORDINATE FIX
import importlib
from utils.gref_pipeline import georef

importlib.reload(georef)
print("✅ Module reloaded - GLOBAL COORDINATE FIX APPLIED!")
print(f"   Module location: {georef.__file__}")
print("\n🔧 Fixed:")
print("   - ✅ Track coordinates: GLOBAL (converted from local with track_offset)")
print("   - ✅ Slit coordinates: GLOBAL (requested bounds, not clipped bounds)")
print("   - ✅ Extent shows what YOU requested, even if crop was clipped at edges")
print("\n📝 Now test with crop_center_slit values that go outside image bounds!")

✅ Module reloaded - GLOBAL COORDINATE FIX APPLIED!
   Module location: e:\mjosa_complete\gref4hsi\gref4hsi\final_act\utils\gref_pipeline\georef.py

🔧 Fixed:
   - ✅ Track coordinates: GLOBAL (converted from local with track_offset)
   - ✅ Slit coordinates: GLOBAL (requested bounds, not clipped bounds)
   - ✅ Extent shows what YOU requested, even if crop was clipped at edges

📝 Now test with crop_center_slit values that go outside image bounds!


In [3]:
# Load 028 transect
transect = load_transect(r"E:\mjosa_new_oct_2025\use_gref4hsi\028\output")
transect.list_files()

# Select 028_5 for NDI analysis
cube = transect.select_files(
    [
        "rad_uhi_20241029_125028_1",
        "rad_uhi_20241029_125028_2",
        "rad_uhi_20241029_125028_3",
        "rad_uhi_20241029_125028_4",
        "rad_uhi_20241029_125028_5",
    ]
)
# cube.describe()


📋 Files in output (corrected=False):
 1. rad_uhi_20241029_125028_1  | shape=(1329, 968, 210)  | has_georef=True
 2. rad_uhi_20241029_125028_2  | shape=(2011, 968, 210)  | has_georef=True
 3. rad_uhi_20241029_125028_3  | shape=(2282, 968, 210)  | has_georef=True
 4. rad_uhi_20241029_125028_4  | shape=(2457, 968, 210)  | has_georef=True
 5. rad_uhi_20241029_125028_5  | shape=(1948, 968, 210)  | has_georef=True
🔄 Rebuilding grids from georef hits for selected files...
   • rad_uhi_20241029_125028_1
  rad_uhi_20241029_125028_1: Using GRIDDED format (T=1329, S=968)
   • rad_uhi_20241029_125028_2
  rad_uhi_20241029_125028_2: Using GRIDDED format (T=2011, S=968)
   • rad_uhi_20241029_125028_3
  rad_uhi_20241029_125028_3: Using GRIDDED format (T=2282, S=968)
   • rad_uhi_20241029_125028_4
  rad_uhi_20241029_125028_4: Using GRIDDED format (T=2457, S=968)
   • rad_uhi_20241029_125028_5
  rad_uhi_20241029_125028_5: Using GRIDDED format (T=1948, S=968)
✅ Combined shapes: X/Y/Z (10027, 968), RGB (

In [4]:
# %matplotlib qt

In [5]:
cube.apply_illumination_correction_v2()

🔄 Using V2 algorithm (pandas rolling median, memory-efficient)
🔄 Computing illumination correction: rolling (window=500), strength=1.0
💾 Memory-efficient mode: Processing file-by-file, direct disk write
📊 Phase 1/2: Computing reference statistics...


   Computing refs:   0%|          | 22/203280 [00:11<31:36:12,  1.79slit-band/s]

KeyboardInterrupt: 

In [ ]:
cube.import_rois("./ROIs/028_new.json")
cube.list_rois()

roi_1 = [
    "1_halo",
    "1_arms",
    "1_dark",
    "1_bomb",
    "sediment",
]
roi_2 = [
    "2_halo",
    "rust",
    "sediment",
    "2_dark",
    "2_bomb",
]
roi_3 = [
    "3_halo",
    "3_dark",
    "sediment",
    "3_bomb",
]
roi_pit = [
    "sediment",
    "dark_pits",
]

roi_halo = [
    "sediment",
    "1_halo",
    "2_halo",
    "3_halo",
]

roi_dark = [
    "sediment",
    "1_dark",
    "2_dark",
    "3_dark",
]

roi_bomb = [
    "sediment",
    "1_bomb",
    "2_bomb",
    "3_bomb",
]


roi_training_binary = [
    "sediment",
    "rust",
]

roi_training = [
    "sediment",
    "rust",
    "1_dark",
    "2_dark",
    "3_dark",
    "1_halo",
    "2_halo",
    "3_halo",
]

# 🔍 Diagnostic: Check RGB Min/Max Values

Before we fix the normalization, let's see what min/max values the code is currently using!

In [ ]:
def check_rgb_minmax(
    cube, use_corrected=True, red_wl=620.0, green_wl=565.0, blue_wl=490.0
):
    """
    Diagnostic function to check current RGB min/max values used for normalization.

    This shows you what the current normalization is doing across ALL loaded files.
    """
    import numpy as np

    # Get the data cube
    cube_data = (
        cube.data_corrected
        if (use_corrected and hasattr(cube, "data_corrected"))
        else cube.data
    )

    # Find wavelength indices
    red_idx = np.argmin(np.abs(cube.wavelengths - red_wl))
    green_idx = np.argmin(np.abs(cube.wavelengths - green_wl))
    blue_idx = np.argmin(np.abs(cube.wavelengths - blue_wl))

    actual_red_wl = cube.wavelengths[red_idx]
    actual_green_wl = cube.wavelengths[green_idx]
    actual_blue_wl = cube.wavelengths[blue_idx]

    # Extract RGB channels (same as plot_rgb does)
    R = cube_data[:, :, red_idx].T.copy()
    G = cube_data[:, :, green_idx].T.copy()
    B = cube_data[:, :, blue_idx].T.copy()

    print("=" * 80)
    print("📊 RGB MIN/MAX DIAGNOSTIC")
    print("=" * 80)
    print(f"\n🔬 Data source: {'data_corrected' if use_corrected else 'data'}")
    print(f"   Cube shape: {cube_data.shape} (tracks × slits × wavelengths)")
    print(f"   Total pixels: {cube_data.shape[0] * cube_data.shape[1]:,}")

    print(f"\n🌈 Wavelengths selected:")
    print(
        f"   Red:   {red_wl:.1f} nm → actual {actual_red_wl:.1f} nm (index {red_idx})"
    )
    print(
        f"   Green: {green_wl:.1f} nm → actual {actual_green_wl:.1f} nm (index {green_idx})"
    )
    print(
        f"   Blue:  {blue_wl:.1f} nm → actual {actual_blue_wl:.1f} nm (index {blue_idx})"
    )

    print(f"\n📈 CURRENT MIN/MAX (used for normalization):")
    print(
        f"   Red channel:   min={R.min():.6f}, max={R.max():.6f}, range={R.max()-R.min():.6f}"
    )
    print(
        f"   Green channel: min={G.min():.6f}, max={G.max():.6f}, range={G.max()-G.min():.6f}"
    )
    print(
        f"   Blue channel:  min={B.min():.6f}, max={B.max():.6f}, range={B.max()-B.min():.6f}"
    )

    # Show percentiles (useful for understanding distribution)
    print(f"\n📊 PERCENTILES (helps understand outliers):")
    for ch_name, ch_data in [("Red", R), ("Green", G), ("Blue", B)]:
        p1 = np.nanpercentile(ch_data, 1)
        p5 = np.nanpercentile(ch_data, 5)
        p50 = np.nanpercentile(ch_data, 50)
        p95 = np.nanpercentile(ch_data, 95)
        p99 = np.nanpercentile(ch_data, 99)
        print(
            f"   {ch_name:5s}: 1%={p1:.4f} | 5%={p5:.4f} | 50%={p50:.4f} | 95%={p95:.4f} | 99%={p99:.4f}"
        )

    # File-by-file breakdown (if multiple files loaded)
    if hasattr(cube, "file_boundaries") and len(cube.file_boundaries) > 1:
        print(f"\n📁 PER-FILE BREAKDOWN:")
        print(f"   Number of files loaded: {len(cube.file_boundaries)}")

        for i, boundary in enumerate(cube.file_boundaries):
            file_name = boundary.get("name", f"File {i+1}")
            start_track = boundary["start_track"]

            # Get end track (next boundary start, or end of data)
            if i < len(cube.file_boundaries) - 1:
                end_track = cube.file_boundaries[i + 1]["start_track"]
            else:
                end_track = cube_data.shape[0]

            # Convert to transposed indices (R, G, B are transposed!)
            # R, G, B have shape (slits, tracks) after transpose
            # So we need to slice along axis 1
            R_file = R[:, start_track:end_track]
            G_file = G[:, start_track:end_track]
            B_file = B[:, start_track:end_track]

            print(f"\n   {file_name}:")
            print(
                f"      Tracks: {start_track} to {end_track} ({end_track - start_track} tracks)"
            )
            print(f"      Red:   min={R_file.min():.6f}, max={R_file.max():.6f}")
            print(f"      Green: min={G_file.min():.6f}, max={G_file.max():.6f}")
            print(f"      Blue:  min={B_file.min():.6f}, max={B_file.max():.6f}")

    print("\n" + "=" * 80)
    print(
        "💡 TIP: If different files have very different min/max, that's why brightness"
    )
    print(
        "   looks inconsistent! Use these values to set manual vmin/vmax per channel."
    )
    print("=" * 80)

    # Return the values in case you want to use them
    return {
        "red": {"min": R.min(), "max": R.max(), "data": R},
        "green": {"min": G.min(), "max": G.max(), "data": G},
        "blue": {"min": B.min(), "max": B.max(), "data": B},
    }


# Example usage:
# minmax_info = check_rgb_minmax(cube, use_corrected=True)

In [ ]:
# 🔍 Run the diagnostic to see current min/max values
minmax_info = check_rgb_minmax(cube, use_corrected=True)

## 🎯 How to Use Manual vmin/vmax

Now that you can see the min/max values, you have **3 options**:

### **Option 1: Single value for all channels**
```python
cube.plot_rgb(
    use_corrected=True,
    vmin=0.5,  # ← Same min for R, G, B
    vmax=1.5,  # ← Same max for R, G, B
    ...
)
```

### **Option 2: Per-channel values (most control!)**
```python
cube.plot_rgb(
    use_corrected=True,
    vmin=(0.4, 0.5, 0.3),  # ← (red_min, green_min, blue_min)
    vmax=(1.5, 1.6, 1.4),  # ← (red_max, green_max, blue_max)
    ...
)
```

### **Option 3: Use specific file's range**
```python
# After running check_rgb_minmax(), get the values for 028_1:
# Then use those values for consistent brightness!
cube.plot_rgb(
    use_corrected=True,
    vmin=(R_min_028_1, G_min_028_1, B_min_028_1),
    vmax=(R_max_028_1, G_max_028_1, B_max_028_1),
    ...
)
```

---

## 📋 Summary: RGB Normalization Fix

### **What was the problem?**
- When loading multiple files (028_1 through 028_5), RGB normalization used **global min/max across all files**
- If 028_1 was brighter than others, its brightness got compressed → appeared darker
- Even after illumination correction, different files can have different absolute intensity ranges

### **The solution:**
✅ **Manual vmin/vmax control** - You can now specify the normalization range!

### **How to fix your plots:**

**Step 1:** Run the diagnostic (cell above) to see current min/max values  
**Step 2:** Copy the values for the file/range you want (e.g., 028_1)  
**Step 3:** Use those values in your plot_rgb() calls  

### **Example workflow:**
```python
# 1. Check what's happening
minmax_info = check_rgb_minmax(cube, use_corrected=True)

# 2. Use values from specific file (shown in diagnostic output)
# Let's say 028_1 has: Red=[0.5, 1.2], Green=[0.6, 1.3], Blue=[0.4, 1.1]

# 3. Apply to all your plots for consistent brightness
cube.plot_rgb(
    use_corrected=True,
    vmin=(0.5, 0.6, 0.4),  # From 028_1
    vmax=(1.2, 1.3, 1.1),  # From 028_1
    ...  # rest of your parameters
)
```

### **When to use each option:**

- **No vmin/vmax:** Auto-brightness per display (crops look good, full transect may vary)
- **Single vmin/vmax:** Quick fix, same range for R, G, B
- **Per-channel vmin/vmax:** Best control, respects channel differences
- **File-specific values:** Consistent brightness when comparing regions from different files

---

In [ ]:
# 🧪 TEST: Compare auto normalization vs manual normalization
# This will show you the difference!
%matplotlib inline
# First, plot with AUTO normalization (current behavior - uses global min/max)
print("📊 Plot 1: AUTO normalization (global min/max)")
cube.plot_rgb(
    use_corrected=True,
    # flip_axes=True,
    # flip_horizontal=True,
    # crop_center_track=1258,
    # crop_center_slit=250,
    # crop_width=500,
    # crop_aspect_ratio=4,
    show_file_boundaries=False,
    # title="AUTO normalization (global min/max)",
)

# Then plot with MANUAL normalization using values you got from diagnostic
# Replace these with actual values from your check_rgb_minmax() output!
print("\n📊 Plot 2: MANUAL normalization (set your own range)")
print("⚠️  Update vmin/vmax below with values from diagnostic!")
cube.plot_rgb(
    use_corrected=True,
    # flip_axes=True,
    # flip_horizontal=True,
    # crop_center_track=1258,
    # crop_center_slit=250,
    # crop_width=500,
    # crop_aspect_ratio=4,
    show_file_boundaries=False,
    vmin=(0.53, 0.58, 0.2),  # ← UPDATE THESE with values from diagnostic!
    vmax=(1.35, 1.37, 1.73),  # ← UPDATE THESE with values from diagnostic!
    # title="MANUAL normalization (fixed range)",
)

In [ ]:
# 🔍 DIAGNOSTIC: Check coordinate system
print("=" * 70)
print("📊 CUBE COORDINATE SYSTEM DIAGNOSTIC")
print("=" * 70)

print(f"\n1️⃣ Cube Properties:")
print(f"   Cube type: {type(cube).__name__}")
print(f"   track_offset: {getattr(cube, 'track_offset', 'NOT FOUND')}")
print(f"   Data shape: {cube.data.shape if hasattr(cube, 'data') else 'N/A'}")
print(f"   Corrected shape: {cube.data_corrected.shape}")

if hasattr(cube, "file_boundaries"):
    print(f"\n2️⃣ File Boundaries:")
    for i, boundary in enumerate(cube.file_boundaries):
        print(f"   File {i+1}: {boundary}")
else:
    print(f"\n2️⃣ File Boundaries: NOT FOUND (single file?)")

print(f"\n3️⃣ Test Crop Parameters:")
print(f"   crop_center_track = 1258 (GLOBAL)")
print(f"   crop_center_slit = 400 (GLOBAL)")
print(f"   crop_width = 200")
print(f"   crop_aspect_ratio = 5")

# Calculate what SHOULD happen
half_width_slit = 200 // 2  # 100
half_width_track = int(200 / 5 / 2)  # 20

track_offset = getattr(cube, "track_offset", 0)
crop_center_track_local = 1258 - track_offset

print(f"\n4️⃣ Expected Crop Bounds (GLOBAL coordinates):")
print(
    f"   Track center: 1258 (global) → {crop_center_track_local} (local, offset={track_offset})"
)
print(
    f"   Track range: [{1258 - half_width_track}, {1258 + half_width_track}] (global)"
)
print(
    f"   Track range: [{crop_center_track_local - half_width_track}, {crop_center_track_local + half_width_track}] (local)"
)
print(
    f"   Slit range: [{400 - half_width_slit}, {400 + half_width_slit}] (should be global)"
)

print(f"\n5️⃣ With flip_axes=True, Axis Labels Should Show:")
print(f"   X-axis (slit): {400 - half_width_slit} to {400 + half_width_slit}")
print(f"   Y-axis (track): {1258 - half_width_track} to {1258 + half_width_track}")
print("=" * 70)

In [ ]:
stop

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    figsize=(5, 50),
    show_file_boundaries=False,
    roi_collection="all",
    roi_legend_loc="outside",
    roi_marker_size=2,
    roi_legend_markersize=50,
    roi_marker_edgewidth=0,
)

In [ ]:
%matplotlib qt
cube.plot_georef(
    apply_alignment_shift=True,
    use_corrected=True,
    coordinate_system="NED",
    figsize=(40, 10),
    # track_start=8980,
    # track_end=9565,
)

In [ ]:
%matplotlib inline


In [ ]:
cube.plot_georef(
    apply_alignment_shift=True,
    use_corrected=False,
    coordinate_system="NED",
    figsize=(40, 10),
    track_start=8980,
    track_end=9565,
)

In [ ]:
# 🔍 Diagnostic: Check which ROIs fall within crop boundaries
def check_rois_in_crop(
    cube,
    roi_list,
    crop_center_track,
    crop_center_slit,
    crop_width,
    crop_aspect_ratio=3.5,
):
    """Check which ROIs from a list fall within the specified crop region."""
    half_width_slit = crop_width // 2
    half_width_track = int(crop_width / crop_aspect_ratio / 2)

    crop_track_min = crop_center_track - half_width_track
    crop_track_max = crop_center_track + half_width_track
    crop_slit_min = crop_center_slit - half_width_slit
    crop_slit_max = crop_center_slit + half_width_slit

    print(
        f"📦 Crop bounds: track [{crop_track_min}:{crop_track_max}], slit [{crop_slit_min}:{crop_slit_max}]"
    )
    print(
        f"   Dimensions: {crop_track_max - crop_track_min} tracks × {crop_slit_max - crop_slit_min} slits\n"
    )

    for roi_name in roi_list:
        if roi_name in cube.roi_collection:
            pixels = cube.roi_collection[roi_name]
            # Filter pixels within crop
            in_crop = [
                (slit, track)
                for slit, track in pixels
                if crop_slit_min <= slit < crop_slit_max
                and crop_track_min <= track < crop_track_max
            ]
            if in_crop:
                tracks = [t for s, t in in_crop]
                slits = [s for s, t in in_crop]
                print(f"✅ {roi_name}: {len(in_crop)} pixels in crop")
                print(f"   Track range: [{min(tracks)}, {max(tracks)}]")
                print(f"   Slit range: [{min(slits)}, {max(slits)}]")
            else:
                # Show where the ROI actually is
                all_tracks = [t for s, t in pixels]
                all_slits = [s for s, t in pixels]
                print(f"❌ {roi_name}: 0 pixels in crop (ROI is outside crop bounds)")
                print(f"   ROI track range: [{min(all_tracks)}, {max(all_tracks)}]")
                print(f"   ROI slit range: [{min(all_slits)}, {max(all_slits)}]")
        else:
            print(f"⚠️  {roi_name}: not found in roi_collection")
        print()


# Example usage:
# check_rois_in_crop(cube, roi_3, 5592, 765, 500, 3.5)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    figsize=(8, 8),
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    figsize=(8, 8),
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    roi_collection=roi_3,
    roi_legend_loc="outside",
    roi_marker_size=200,
    roi_legend_markersize=10,
    roi_marker_edgewidth=0,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
)
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=150,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=1,
    roi_collection=roi_2,
    roi_legend_loc="outside",
    roi_marker_size=200,
    roi_legend_markersize=10,
    roi_marker_edgewidth=0,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=250,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=4,
    display_aspect_ratio=4.0,  # Controls display stretching
)
cube.plot_rgb(
    crop_aspect_ratio=4,
    display_aspect_ratio=4.0,  # Controls display stretching
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=250,
    crop_width=500,
    show_file_boundaries=False,
    roi_collection=roi_2,
    roi_legend_loc="outside",
    roi_marker_size=5,
    roi_legend_markersize=20,
    roi_marker_edgewidth=0,
)

In [ ]:
# 🔍 Check which roi_2 ROIs are in this crop region
check_rois_in_crop(cube, roi_2, 1258, 212, 400, 3.5)

In [ ]:
track = 1258
slit = 250


cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=track,
    crop_center_slit=slit,
    crop_width=500,
    show_file_boundaries=False,
    crop_aspect_ratio=4,
    display_aspect_ratio=4.0,  # Controls display stretching
)
cube.plot_rgb(
    crop_aspect_ratio=4,
    display_aspect_ratio=4.0,  # Controls display stretching
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=track,
    crop_center_slit=slit,
    crop_width=500,
    show_file_boundaries=False,
    roi_collection=roi_2,
    roi_legend_loc="outside",
    roi_marker_size=5,
    roi_legend_markersize=20,
    roi_marker_edgewidth=0,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=5160,
    crop_center_slit=613,
    show_file_boundaries=False,
    crop_width=500,
    crop_aspect_ratio=4,
    display_aspect_ratio=4.0,  # Controls display stretching
    roi_collection=roi_1,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=717,
    crop_center_slit=385,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
)

In [ ]:
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    # figsize=(3, 10),
    flip_horizontal=True,
    crop_center_track=717,
    crop_center_slit=385,
    crop_width=400,
    show_file_boundaries=False,
    crop_aspect_ratio=3.5,
    roi_collection=roi_pit,
)

In [ ]:
# cube.plot_rgb(
#     use_corrected=True,
#     flip_axes=True,
#     # figsize=(3, 10),
#     flip_horizontal=True,
#     crop_center_track=4026,
#     crop_center_slit=582,
#     crop_width=400,
#     show_file_boundaries=False,
#     crop_aspect_ratio=3.5,
# )

---

---

---

---

---

---

# check spectra

In [ ]:
%matplotlib qt

In [ ]:
use_roi = "all"
# use_roi = roi_1
# use_roi = roi_3
# use_roi = roi_pit
# use_roi = roi_bomb
# use_roi = roi_halo

cube.plot_spectrum(
    roi_names=use_roi,
    use_corrected=True,
    wavelength_range=(490, 700),
    wavelength_smoothing=10,
    smoothing_method="gaussian",
    gaussian_sigma=5,
    legend_loc="outside",
    use_inline_labels=False,
    show_std=False,
    smooth_before_filter=True,
    # normalize_method="l2",
    # derivative_order=1,
)

cube.plot_spectrum(
    roi_names=use_roi,
    use_corrected=True,
    wavelength_range=(490, 700),
    wavelength_smoothing=10,
    smoothing_method="gaussian",
    gaussian_sigma=5,
    legend_loc="outside",
    use_inline_labels=False,
    show_std=False,
    smooth_before_filter=True,
    normalize_method="l2",
    # derivative_order=1,
)

---

---

---

---

---

## ✅ MNF Analysis Summary

**What you now have:**

1. **`cube.mnf_data`** - Full MNF-transformed hyperspectral cube (10k+ tracks × 1024 slits × 20 components)
2. **`plot_mnf_rgb()`** - Function to visualize MNF as RGB composite with cropping + ROI overlay support
3. **Eigenvalue plots** - Shows which components have strongest signal

**Next steps you can try:**

### 📊 Option 1: Analyze which MNF components separate your features
- Extract MNF values for each ROI
- Plot MNF "spectra" (component 1-20 values) for rust vs sediment vs bombs
- See which components best distinguish your classes

### 🎯 Option 2: Use MNF for classification
- Train classifier on top 3-5 MNF components instead of all 120 wavelengths
- Faster training + less overfitting + better generalization

### 🔍 Option 3: Individual component analysis
- Plot each MNF component as individual grayscale heatmap
- See which specific component highlights rust, bombs, or dark features

**Pro tip:** If features don't separate well in MNF1-3, try looking at MNF4-6. Sometimes specific features show up in mid-level components!

---

## 🔍 How to Evaluate MNF Quality

### **What MNF Components Actually Are:**

MNF components are **NOT wavelengths**! They are **synthetic features** created by combining all wavelengths.

- **Original data:** 120 wavelengths → intensity vs wavelength → physical units (radiance)
- **MNF data:** 20 components → unitless scores → NO wavelength axis!

**You CANNOT plot MNF as a "spectrum"** because there's no wavelength dimension. Instead:

### **1. Check Eigenvalue Plot** (variance explained)
✅ **Good MNF:** Top 3 components > 70% variance, sharp drop after  
❌ **Poor MNF:** Top 3 components < 50% variance, slow gradual drop

### **2. Visual Comparison** (MNF vs Raw RGB)
Run the side-by-side comparison below to see if MNF makes features clearer!

### **3. ROI Separability** (if features are distinct)
Extract MNF scores for each ROI and check if they cluster separately (see analysis cells below)

In [ ]:
# 🔬 Step 1: Compute MNF on full transect
# This takes 1-2 minutes but only needs to be done once!
# Run this cell first, then run the cells below for visualization
print("⏳ Computing MNF - this will take 1-2 minutes...\n")
cube.apply_mnf_transform(use_corrected=True, n_components=20, quiet=False)

In [ ]:
# 📊 Side-by-side comparison: Raw RGB vs MNF RGB (same crop region)
# NOTE: These will appear as separate plots, not side-by-side
# (plot_rgb and plot_mnf_rgb don't support custom axes)

print("🖼️  Showing Raw RGB first, then MNF RGB below...\n")

# FIRST: Raw wavelength RGB
cube.plot_rgb(
    use_corrected=True,
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    crop_aspect_ratio=3.5,
    roi_collection=roi_3,
    roi_marker_size=200,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    figsize=(8, 8),  # Square figure
    title="RAW: Wavelength RGB (635-565-490nm)",
)

# SECOND: MNF RGB (will be same size now)
cube.plot_mnf_rgb(
    components=[1, 2, 3],
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    crop_aspect_ratio=3.5,
    roi_collection=roi_3,
    roi_marker_size=200,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    figsize=(8, 8),  # Same square figure
    title="MNF: Components 1-2-3",
)

print("\n🔍 Compare the two images:")
print("   ✅ Are bombs/rust/dark spots more distinct in MNF?")
print("   ✅ Is there less noise in MNF?")
print("   ✅ Do ROIs separate better visually?")

In [ ]:
# 📈 MNF "Spectra" - Plot component scores per ROI
# (This is NOT intensity vs wavelength - it's score vs component number!)

import numpy as np
import matplotlib.pyplot as plt

# Select ROIs to analyze
rois_to_analyze = ["3_halo", "3_dark", "3_bomb", "sediment"]

fig, ax = plt.subplots(figsize=(12, 6))

for roi_name in rois_to_analyze:
    if roi_name not in cube.roi_collection:
        print(f"⚠️  {roi_name} not found in ROI collection")
        continue

    roi_pixels = cube.roi_collection[roi_name]

    # Extract MNF scores for this ROI
    mnf_scores = []
    for slit, track in roi_pixels:
        # Convert to relative track index
        rel_track = track - cube.track_offset

        if (
            0 <= rel_track < cube.mnf_data.shape[0]
            and 0 <= slit < cube.mnf_data.shape[1]
        ):
            pixel_scores = cube.mnf_data[rel_track, slit, :]
            if np.isfinite(pixel_scores).all():
                mnf_scores.append(pixel_scores)

    if len(mnf_scores) > 0:
        mnf_scores = np.array(mnf_scores)
        mean_scores = np.mean(mnf_scores, axis=0)
        std_scores = np.std(mnf_scores, axis=0)

        # Get color from ROI color mapping
        color = cube._get_roi_color(roi_name, ["yellow", "cyan", "magenta"], None)

        # Plot mean ± std
        components = np.arange(1, len(mean_scores) + 1)
        ax.plot(
            components,
            mean_scores,
            "o-",
            label=f"{roi_name} (n={len(mnf_scores)})",
            color=color,
            linewidth=2,
            markersize=6,
        )
        ax.fill_between(
            components,
            mean_scores - std_scores,
            mean_scores + std_scores,
            alpha=0.2,
            color=color,
        )

ax.set_xlabel("MNF Component Number", fontsize=12)
ax.set_ylabel("MNF Score (unitless)", fontsize=12)
ax.set_title(
    "MNF Component Scores per ROI (Mean ± Std)", fontsize=14, fontweight="bold"
)
ax.legend(loc="best", fontsize=10)
ax.grid(alpha=0.3)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("\n🔍 Interpretation:")
print("   - X-axis = MNF component number (NOT wavelength!)")
print("   - Y-axis = MNF score (unitless, standard deviations from mean)")
print("   - High score = pixel is STRONG in that pattern")
print("   - Low score = pixel is WEAK in that pattern")
print("\n✅ Good separation = Different ROIs have different score patterns")
print("❌ Poor separation = All ROIs look similar across all components")

In [ ]:
# 🎯 MNF Scatter Plot - Quantitative ROI Separability Test
# Plot MNF1 vs MNF2 to see if ROIs cluster separately

import numpy as np
import matplotlib.pyplot as plt

rois_to_analyze = ["3_halo", "3_dark", "3_bomb", "sediment"]

fig, ax = plt.subplots(figsize=(10, 8))

for roi_name in rois_to_analyze:
    if roi_name not in cube.roi_collection:
        continue

    roi_pixels = cube.roi_collection[roi_name]

    mnf1_values = []
    mnf2_values = []

    for slit, track in roi_pixels:
        rel_track = track - cube.track_offset

        if (
            0 <= rel_track < cube.mnf_data.shape[0]
            and 0 <= slit < cube.mnf_data.shape[1]
        ):
            mnf1 = cube.mnf_data[rel_track, slit, 0]  # MNF1
            mnf2 = cube.mnf_data[rel_track, slit, 1]  # MNF2

            if np.isfinite(mnf1) and np.isfinite(mnf2):
                mnf1_values.append(mnf1)
                mnf2_values.append(mnf2)

    if len(mnf1_values) > 0:
        color = cube._get_roi_color(roi_name, ["yellow", "cyan", "magenta"], None)
        ax.scatter(
            mnf1_values,
            mnf2_values,
            alpha=0.6,
            s=50,
            label=f"{roi_name} (n={len(mnf1_values)})",
            color=color,
            edgecolors="black",
            linewidths=0.5,
        )

ax.set_xlabel("MNF Component 1 Score", fontsize=12)
ax.set_ylabel("MNF Component 2 Score", fontsize=12)
ax.set_title("MNF1 vs MNF2: ROI Clustering", fontsize=14, fontweight="bold")
ax.legend(loc="best", fontsize=10)
ax.grid(alpha=0.3)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("\n🔍 Interpretation:")
print("   ✅ Good MNF: ROIs form distinct clusters (well separated)")
print("   ❌ Poor MNF: ROIs overlap heavily (not distinguishable)")
print("\nℹ️  If clusters overlap, try:")
print("   - Plot MNF1 vs MNF3, or MNF2 vs MNF3")
print(
    "   - Use top 3-5 MNF components for classification instead of all 120 wavelengths"
)

---

## 📝 Summary: MNF vs Wavelength Spectra

### **Key Differences:**

| Aspect | **Wavelength Spectra** | **MNF Components** |
|--------|----------------------|-------------------|
| **X-axis** | Wavelength (nm) | Component number (1-20) |
| **Y-axis** | Intensity (radiance) | Unitless score (std dev) |
| **Meaning** | Physical light intensity at each wavelength | How strongly pixel exhibits each pattern |
| **Units** | W·m⁻²·sr⁻¹·nm⁻¹ | Dimensionless |
| **Interpretation** | "How much red/green/blue light?" | "How rusty/dark/textured is this?" |
| **Plot type** | `cube.plot_spectrum()` | Component scores (line plot) |

### **When to Use Each:**

**Use Wavelength Spectra when:**
- ✅ Identifying materials by absorption features (e.g., chlorophyll at 680nm)
- ✅ Understanding physical properties (reflectance, absorption)
- ✅ Comparing to lab spectra or spectral libraries

**Use MNF Components when:**
- ✅ Reducing noise before classification
- ✅ Finding unknown patterns in complex data
- ✅ Reducing dimensionality (120 bands → 5 components)
- ✅ Visualizing multi-dimensional data as RGB

### **Bottom Line:**
MNF doesn't replace spectral analysis—it **complements** it! Use spectra to understand **what** your features are (rust, sediment, algae), then use MNF to **find and classify** them more accurately.

In [ ]:
# 🧪 Experiment with different MNF combinations
# Change components=[1,2,3] to try different combinations!

# Example: Try MNF 1, 3, 4 (skip MNF2)
cube.plot_mnf_rgb(
    components=[1, 3, 4],  # ← Change this!
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    crop_aspect_ratio=3.5,
    roi_collection=roi_3,
    roi_marker_size=200,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=10,
    title="Experiment: MNF1-MNF3-MNF4",
)

## 🔬 Experiment: Try Different MNF Component Combinations

Different MNF components highlight different features! Try these combinations:

- **[1, 2, 3]** - Default (usually best overall)
- **[1, 3, 4]** - Skip MNF2, see if other features pop
- **[2, 3, 4]** - Skip MNF1, see secondary patterns
- **[1, 1, 2]** - Emphasize MNF1 (red + green = yellow if high)

**What to look for:**
- Do rust/bombs/dark spots separate better in certain combinations?
- Which component makes your ROIs most visible?

In [ ]:
# 🖼️ Full transect MNF view (no crop) - overview of all features
cube.plot_mnf_rgb(
    components=[1, 2, 3],
    flip_axes=True,
    flip_horizontal=True,
    roi_collection="all",
    figsize=(5, 50),
    roi_marker_size=2,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=50,
    title="Full Transect: MNF1-MNF2-MNF3 (All ROIs)",
)

In [ ]:
# 🖼️ Crop 3: MNF visualization with ROI overlay (roi_1)
cube.plot_mnf_rgb(
    components=[1, 2, 3],
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=5160,
    crop_center_slit=613,
    crop_width=400,
    crop_aspect_ratio=3.5,
    roi_collection=roi_1,
    roi_marker_size=200,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=10,
    title="Crop 3: MNF1-MNF2-MNF3 (1_halo, 1_arms, 1_dark, 1_bomb)",
)

In [ ]:
# 🖼️ Crop 2: MNF visualization with ROI overlay (roi_2 - includes rust!)
cube.plot_mnf_rgb(
    components=[1, 2, 3],
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=1258,
    crop_center_slit=212,
    crop_width=400,
    crop_aspect_ratio=3.5,
    roi_collection=roi_2,
    roi_marker_size=200,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=10,
    title="Crop 2: MNF1-MNF2-MNF3 (2_halo, rust, 2_dark)",
)

In [ ]:
# 🖼️ Crop 1: MNF visualization with ROI overlay (roi_3)
cube.plot_mnf_rgb(
    components=[1, 2, 3],  # Try [1,2,3], [1,3,4], [2,3,4] to see different features
    flip_axes=True,
    flip_horizontal=True,
    crop_center_track=5592,
    crop_center_slit=765,
    crop_width=500,
    crop_aspect_ratio=3.5,
    roi_collection=roi_3,
    roi_marker_size=200,
    roi_legend_loc="outside",
    roi_marker_edgewidth=0,
    roi_legend_markersize=10,
    title="Crop 1: MNF1-MNF2-MNF3 (3_halo, 3_dark, 3_bomb)",
)

## 🖼️ MNF Visualization Examples

Now let's visualize your crop regions using MNF components instead of raw wavelengths!

**What to expect:**
- Features (rust, bombs, dark spots) should "pop out" more clearly
- Different features may show up in different MNF components
- Less noise compared to raw RGB wavelengths

# 🔬 MNF (Minimum Noise Fraction) Analysis

## What is MNF?

**MNF creates synthetic "super-bands" by combining all wavelengths to maximize signal-to-noise ratio.**

### Your Original Data:
- 120 wavelength bands (490nm → 680nm)
- Each pixel = `[intensity_490nm, intensity_495nm, ..., intensity_680nm]`

### After MNF:
- 20 MNF components (MNF1, MNF2, ..., MNF20)
- Each pixel = `[MNF1_value, MNF2_value, ..., MNF20_value]`

### Key Points:
- **MNF1** = Best signal-to-noise pattern (distinct features with minimal noise)
- **MNF2** = Second-best pattern (different features than MNF1)
- **MNF3** = Third-best pattern, etc.
- **MNF50+** = Mostly noise (can be discarded)

### Why Use MNF?
✅ Makes features "pop out" more clearly than raw wavelengths  
✅ Reduces noise automatically  
✅ Helps identify which spectral patterns distinguish rust/bombs/dark spots  
✅ Top 3 components usually capture 90%+ of useful information

---

## Methods Now Available:
```python
# Compute MNF transformation
cube.apply_mnf_transform(use_corrected=True, n_components=20)

# Visualize MNF as RGB composite
cube.plot_mnf_rgb(components=[1,2,3], flip_axes=True, flip_horizontal=True,
                  crop_center_track=5592, crop_center_slit=765, crop_width=500,
                  roi_collection=roi_3)
```

In [ ]:
# 📊 Visualize MNF eigenvalues (signal strength of each component)
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Eigenvalues (signal strength)
ax1.bar(
    range(1, len(cube.mnf_eigenvalues) + 1), cube.mnf_eigenvalues, color="steelblue"
)
ax1.set_xlabel("MNF Component", fontsize=12)
ax1.set_ylabel("Eigenvalue (Signal Strength)", fontsize=12)
ax1.set_title("MNF Component Signal Strength", fontsize=14, fontweight="bold")
ax1.grid(alpha=0.3)
ax1.set_xticks(range(1, len(cube.mnf_eigenvalues) + 1))

# Highlight top 3
for i in range(3):
    ax1.get_children()[i].set_color("darkgreen")
    ax1.get_children()[i].set_alpha(0.8)

# Plot 2: Cumulative explained variance
ax2.plot(
    range(1, len(cube.mnf_cumulative_variance) + 1),
    cube.mnf_cumulative_variance * 100,
    marker="o",
    linewidth=2,
    markersize=6,
    color="darkgreen",
)
ax2.axhline(y=90, color="red", linestyle="--", alpha=0.7, label="90% threshold")
ax2.set_xlabel("Number of MNF Components", fontsize=12)
ax2.set_ylabel("Cumulative Explained Variance (%)", fontsize=12)
ax2.set_title("How Many Components Do We Need?", fontsize=14, fontweight="bold")
ax2.grid(alpha=0.3)
ax2.legend()
ax2.set_xticks(range(1, len(cube.mnf_cumulative_variance) + 1))

plt.tight_layout()
plt.show()

# Print summary
print(f"\n📊 MNF Component Summary:")
print(f"=" * 60)
for i in range(min(10, len(cube.mnf_explained_variance))):
    print(
        f"   MNF{i+1:2d}: {cube.mnf_explained_variance[i]*100:5.1f}% variance | "
        f"Cumulative: {cube.mnf_cumulative_variance[i]*100:5.1f}%"
    )
print(f"=" * 60)
print(
    f"\n💡 Insight: Top 3 components explain {cube.mnf_cumulative_variance[2]*100:.1f}% of useful signal!"
)

## 🖼️ MNF Visualization Examples

Now let's visualize your crop regions using MNF components instead of raw wavelengths!

**What to expect:**
- Features (rust, bombs, dark spots) should "pop out" more clearly
- Different features may show up in different MNF components
- Less noise compared to raw RGB wavelengths